In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H11b — BPR Triple Variation: Confidence vs MFLS vs QuadSurf
# ══════════════════════════════════════════════════════════════════════
# Three continuous-prior strategies for BPR flip selection, all derived
# from the BSDT continuous particle positions s ∈ [-1,1]ⁿ:
#
# 1. CONFIDENCE:  w(v) = 1 - |s_v|
#    Spatial uncertainty. Variables near 0 = ambiguous = cheap to flip.
#    (Original BPR concept — no other SLS solver has this.)
#
# 2. MFLS (Multi-Factor Latent Score):  w(v) = |∂E_SAT/∂s_v| / max
#    Gradient magnitude of SAT energy at continuous position.
#    High gradient = variable in transition = optimizer couldn't settle.
#    Adapted from BSDT framework: MFLS = ‖∇E_BS‖_F (system_mode.py)
#
# 3. QUADSURF (Quadratic Surface):  w(v) = [Σ_{c∋v} E_c] / max
#    Per-variable clause energy sum. Measures how much "energy surface"
#    this variable sits on. High = participates in many nearly-unsat
#    clauses = critical surface boundary = strategic flip target.
#    Adapted from Morse theory / Hessian analysis (ReducedTensorDescriptor)
#
# ALL weights normalised to [0,1] and used as TIEBREAKERS:
#    score(v) = exp(-(break-make)/T) × (1 + β·w(v))     β=0.3
#
# Key fix from H11: H11 used (conf+0.1) giving 11x multiplier range
# — weight dominated energy. Now β=0.3 gives max 1.3x — energy leads.
#
# Also: max_flips raised 100K → 200K (H11 hit ceiling at α=4.2)
#
# Author: Odeyemi Olusegun Israel
# ══════════════════════════════════════════════════════════════════════
import torch, numpy as np, time
from numba import njit

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# ── Instance generator ────────────────────────────────────────────────
def generate_3sat_instance(n, m):
    vars_idx = torch.randint(0, n, (m, 3))
    signs = torch.randint(0, 2, (m, 3)) * 2 - 1
    return list(zip(vars_idx.tolist(), signs.tolist()))


# ══════════════════════════════════════════════════════════════════════
#  BPR V2 — energy-dominant + tiebreaker weight (Numba)
# ══════════════════════════════════════════════════════════════════════

@njit(cache=True)
def bpr_repair(clauses_v, clauses_s, assignment, weight,
               max_flips=200000, T_init=0.5, T_min=0.01,
               p_random=0.1, beta=0.3):
    """
    BSDT Probabilistic Repair V2 — energy + weighted tiebreaker.

    score(v) = exp(-(break-make)/T) × (1 + β·weight[v])

    Compared to H11:  used (conf+0.1) → 11x range → weight dominated.
    V2 uses (1 + β·w) → 1.3x range → energy leads, weight tiebreaks.

    Parameters
    ----------
    clauses_v : (m, 3) int32
    clauses_s : (m, 3) int32
    assignment : (n,) int32
    weight : (n,) float64  — normalised [0,1] continuous prior
    max_flips : int         — raised to 200K from 100K
    beta : float            — tiebreaker strength (0.3 default)
    """
    m = clauses_v.shape[0]
    n = assignment.shape[0]

    # ── Build flat var→clause adjacency ──
    var_count = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            var_count[clauses_v[c, j]] += 1
    var_off = np.zeros(n + 1, dtype=np.int32)
    for v in range(n):
        var_off[v + 1] = var_off[v] + var_count[v]
    var_adj = np.zeros(var_off[n], dtype=np.int32)
    fill = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            v = clauses_v[c, j]
            var_adj[var_off[v] + fill[v]] = c
            fill[v] += 1

    # ── var→sign-in-clause lookup ──
    var_sign = np.zeros(var_off[n], dtype=np.int32)
    fill2 = np.zeros(n, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            v = clauses_v[c, j]
            var_sign[var_off[v] + fill2[v]] = clauses_s[c, j]
            fill2[v] += 1

    # ── Init clause satisfaction counts ──
    clause_sat = np.zeros(m, dtype=np.int32)
    for c in range(m):
        for j in range(3):
            v = clauses_v[c, j]
            s = clauses_s[c, j]
            if (assignment[v] == 1 and s == 1) or (assignment[v] == 0 and s == -1):
                clause_sat[c] += 1

    # ── Incremental unsatisfied clause list ──
    unsat_list = np.zeros(m, dtype=np.int32)
    unsat_pos = np.full(m, -1, dtype=np.int32)
    n_unsat = 0
    for c in range(m):
        if clause_sat[c] == 0:
            unsat_pos[c] = n_unsat
            unsat_list[n_unsat] = c
            n_unsat += 1

    for flip in range(max_flips):
        if n_unsat == 0:
            return assignment, flip

        ci = unsat_list[np.random.randint(n_unsat)]

        # Cosine-annealed temperature
        progress = flip / max_flips
        T = T_min + (T_init - T_min) * 0.5 * (1.0 + np.cos(3.141592653589793 * progress))

        if np.random.random() < p_random:
            v_flip = clauses_v[ci, np.random.randint(3)]
        else:
            scores = np.zeros(3, dtype=np.float64)

            for j in range(3):
                v_cand = clauses_v[ci, j]
                make = 0
                brk = 0

                for idx in range(var_off[v_cand], var_off[v_cand + 1]):
                    cc = var_adj[idx]
                    s_here = var_sign[idx]
                    satisfies_cc = ((assignment[v_cand] == 1 and s_here == 1) or
                                    (assignment[v_cand] == 0 and s_here == -1))
                    if satisfies_cc:
                        if clause_sat[cc] == 1:
                            brk += 1
                    else:
                        if clause_sat[cc] == 0:
                            make += 1

                delta = brk - make
                # ═══ V2 TIEBREAKER FORMULA ═══
                # Energy term dominates; weight is a mild tiebreaker
                scores[j] = np.exp(-delta / (T + 1e-10)) * (1.0 + beta * weight[v_cand])

            total = scores[0] + scores[1] + scores[2]
            if total < 1e-30:
                v_flip = clauses_v[ci, np.random.randint(3)]
            else:
                r = np.random.random() * total
                if r <= scores[0]:
                    v_flip = clauses_v[ci, 0]
                elif r <= scores[0] + scores[1]:
                    v_flip = clauses_v[ci, 1]
                else:
                    v_flip = clauses_v[ci, 2]

        # ── Execute flip + incremental updates O(degree(v)) ──
        assignment[v_flip] = 1 - assignment[v_flip]

        for idx in range(var_off[v_flip], var_off[v_flip + 1]):
            cc = var_adj[idx]
            s_here = var_sign[idx]
            old_sat = clause_sat[cc]

            now_satisfies = ((assignment[v_flip] == 1 and s_here == 1) or
                             (assignment[v_flip] == 0 and s_here == -1))
            if now_satisfies:
                clause_sat[cc] += 1
            else:
                clause_sat[cc] -= 1

            new_sat = clause_sat[cc]
            if old_sat == 0 and new_sat > 0:
                pos = unsat_pos[cc]
                last = unsat_list[n_unsat - 1]
                unsat_list[pos] = last
                unsat_pos[last] = pos
                unsat_pos[cc] = -1
                n_unsat -= 1
            elif old_sat > 0 and new_sat == 0:
                unsat_list[n_unsat] = cc
                unsat_pos[cc] = n_unsat
                n_unsat += 1

    return assignment, max_flips


# ══════════════════════════════════════════════════════════════════════
#  BSDTGravityV2 + Weight Extractors
# ══════════════════════════════════════════════════════════════════════

class BSDTGravityV2:
    def __init__(self, n, clauses, mu_scale=0.1,
                 G_max=0.10, top_k_frac=0.1, gravity_start=0.2,
                 elite_repulsion=0.5, gravity_interval=20):
        self.n = n
        self.m = len(clauses)
        self.mu_scale = mu_scale
        self.G_max = G_max
        self.top_k_frac = top_k_frac
        self.gravity_start = gravity_start
        self.elite_repulsion = elite_repulsion
        self.gravity_interval = gravity_interval
        self.device = device

        vs_list = [vs for vs, ss in clauses]
        ss_list = [ss for vs, ss in clauses]
        self.vars_t  = torch.tensor(vs_list, dtype=torch.long,    device=device)
        self.signs_t = torch.tensor(ss_list, dtype=torch.float32, device=device)
        self.pos_mask = (self.signs_t > 0).long()

        # BPR clause format (numpy)
        self.clauses_v = np.array(vs_list, dtype=np.int32)
        self.clauses_s = np.array(ss_list, dtype=np.int32)

    def _energy_core(self, s, mu_val, vars_t, signs_t):
        lit = s[:, vars_t] * signs_t.unsqueeze(0)
        e_sat = (torch.prod(1.0 - lit, dim=-1) / 8.0).sum(-1)
        if mu_val > 0:
            return e_sat + mu_val * ((1.0 - s * s) ** 2).sum(-1)
        return e_sat

    def find_best_particle(self, s):
        with torch.no_grad():
            x = (s > 0).long()
            lit_ok = (x[:, self.vars_t] == self.pos_mask.unsqueeze(0))
            n_sat = lit_ok.any(dim=2).sum(dim=1)
            best = n_sat.argmax()
            return best.item(), self.m - n_sat[best].item()

    def check_single(self, x):
        with torch.no_grad():
            n_sat = int((x[self.vars_t] == self.pos_mask).any(dim=1).sum())
            return n_sat == self.m, self.m - n_sat

    # ── Weight extractors (one per variation) ────────────────────────

    def get_confidence(self, s, best_idx):
        """W1 — Confidence: w(v) = 1 - |s_v| (spatial uncertainty).
        Ambiguous vars (|s_v|≈0) get high weight → prefer to flip.
        """
        with torch.no_grad():
            return (1.0 - s[best_idx].abs()).cpu().numpy().astype(np.float64)

    def get_mfls(self, s, best_idx):
        """W2 — MFLS: w(v) = |∂E_SAT/∂s_v| / max  (gradient magnitude).
        From BSDT framework (system_mode.py line 1002):
          MFLS = ‖∇E_BS‖_F
        For SAT: per-variable gradient of clause energy.
        High gradient = variable in transition = optimizer couldn't settle
        = candidate for repair flip.
        """
        sb = s[best_idx].clone().detach().requires_grad_(True)
        # Pure SAT energy (no penalty) — captures clause landscape only
        lit = sb[self.vars_t] * self.signs_t
        e_sat = (torch.prod(1.0 - lit, dim=-1) / 8.0).sum()
        grad = torch.autograd.grad(e_sat, sb)[0]
        g = grad.abs().cpu().numpy().astype(np.float64)
        mx = g.max()
        if mx < 1e-10:
            return np.full(self.n, 0.5, dtype=np.float64)
        return g / mx

    def get_quadsurf(self, s, best_idx):
        """W3 — QuadSurf: w(v) = [Σ_{c∋v} E_c] / max  (clause surface tension).
        From Morse theory / ReducedTensorDescriptor:
          Hessian eigenvalues → saddle point detection
        For SAT: per-variable sum of clause energies.
        High = variable participates in many barely-satisfied clauses
        = sits on the "energy surface" between SAT/UNSAT
        = strategic flip target (analogous to saddle point).
        """
        with torch.no_grad():
            sb = s[best_idx]
            lit = sb[self.vars_t] * self.signs_t   # (m, 3)
            clause_e = (1.0 - lit).prod(dim=1) / 8.0  # (m,) per-clause energy
            qs = torch.zeros(self.n, device=self.device)
            for j in range(3):
                qs.scatter_add_(0, self.vars_t[:, j], clause_e)
            qs = qs.cpu().numpy().astype(np.float64)
            mx = qs.max()
            if mx < 1e-10:
                return np.full(self.n, 0.5, dtype=np.float64)
            return qs / mx

    # ── Gravity flow (identical to H10b) ─────────────────────────────

    def gravity_flow(self, steps=4000, particles=1000,
                     schedule='delay70', lr=0.02):
        n = self.n
        gi = self.gravity_interval
        s = torch.randn(particles, n, device=self.device) * 0.1
        s.requires_grad_(True)
        grav_step  = int(self.gravity_start * steps)
        delay_step = int(0.7 * steps)
        top_k = max(1, int(self.top_k_frac * particles))
        theta = None
        use_amp = (self.device.type == 'cuda')
        cached_target = None

        for step in range(steps):
            if step < delay_step:
                mu = 0.0
            else:
                t_l = (step - delay_step) / (steps - delay_step)
                mu = self.mu_scale * 0.5 * (1.0 - np.cos(np.pi * t_l))

            if use_amp:
                with torch.amp.autocast('cuda'):
                    e = self._energy_core(s, mu, self.vars_t, self.signs_t)
                    e_f32 = e.float()
            else:
                e_f32 = self._energy_core(s, mu, self.vars_t, self.signs_t)

            e_vals = e_f32.detach()
            e_f32.sum().backward()

            with torch.no_grad():
                if theta is None:
                    theta = float(e_vals.median()) + 1e-8
                damp = 1.0 / (1.0 + e_vals.unsqueeze(1) / theta)
                s.sub_(lr * damp * s.grad)

                if step >= grav_step and (step - grav_step) % gi == 0:
                    progress = (step - grav_step) / (steps - grav_step)
                    g = self.G_max * progress * progress * gi
                    _, top_idx = e_vals.topk(top_k, largest=False)
                    elite = s[top_idx]
                    d = torch.cdist(s, elite)
                    cached_target = elite[d.argmin(dim=1)]
                    if top_k > 1:
                        ed = d[top_idx]
                        ed.fill_diagonal_(float('inf'))
                        nn_e = ed.argmin(dim=1)
                        push = elite - elite[nn_e]
                        pn = push.norm(dim=1, keepdim=True).clamp_(min=1e-6)
                        s[top_idx] += (self.elite_repulsion * g) * (push / pn)
                    s.add_(g * (cached_target - s))
                elif step >= grav_step and cached_target is not None:
                    progress = (step - grav_step) / (steps - grav_step)
                    g = self.G_max * progress * progress
                    s.add_(g * (cached_target - s))

                s.clamp_(-1, 1)
                if (step + 1) % 200 == 0:
                    theta = float(e_vals.median()) + 1e-8

            s.requires_grad_(True)
            if s.grad is not None:
                s.grad.zero_()

        return s.detach()


# ── Compile + warmup ──────────────────────────────────────────────────
try:
    BSDTGravityV2._energy_core = torch.compile(BSDTGravityV2._energy_core)
    print('✓ torch.compile applied')
except Exception:
    print('⚠ torch.compile unavailable')

# Warmup BPR (triggers Numba compilation)
_w = np.array([0.5, 0.3, 0.8], dtype=np.float64)
_ = bpr_repair(
    np.array([[0, 1, 2]], dtype=np.int32),
    np.array([[1, -1, 1]], dtype=np.int32),
    np.array([1, 0, 1], dtype=np.int32),
    _w, max_flips=10, beta=0.3
)

print('✓ BPR V2 (tiebreaker) + GravityV2 loaded')
print(f'  Device: {device}')
if device.type == 'cuda':
    print(f'  GPU:    {torch.cuda.get_device_name()}')
print()
print('  Three weight strategies from BSDT continuous positions:')
print('    1. CONFIDENCE:  w(v) = 1 - |s_v|          [spatial uncertainty]')
print('    2. MFLS:        w(v) = |∂E/∂s_v| / max    [gradient magnitude]')
print('    3. QUADSURF:    w(v) = Σ E_c(v) / max     [clause surface tension]')
print()
print('  Scoring: exp(-(break-make)/T) × (1 + 0.3·w)')
print('  max_flips = 200K  |  T: 0.5 → 0.01 cosine anneal')

In [ ]:
# ══════════════════════════════════════════════════════════════════════
# H11b Experiment: Triple BPR Variation (Confidence vs MFLS vs QuadSurf)
# ══════════════════════════════════════════════════════════════════════
# Gravity runs ONCE per instance. All 3 BPR weights tested on same output.
# ══════════════════════════════════════════════════════════════════════
import matplotlib.pyplot as plt

ALPHAS   = [3.8, 4.0, 4.2]
NS       = [500, 750, 1000]
N_INST   = 50
PARTICLES = 1000
STEPS    = 4000
BETA     = 0.3
MAX_FLIPS = 200000

MODES = ['confidence', 'mfls', 'quadsurf']

# Baselines (from previous runs)
h10b = {
    (3.8, 500): 98.0, (3.8, 750): 98.0, (3.8, 1000): 96.0,
    (4.0, 500): 72.0, (4.0, 750): 60.0, (4.0, 1000): 30.0,
    (4.2, 500): 10.0, (4.2, 750):  6.0, (4.2, 1000):  0.0,
}
h9b = {
    (3.8, 500): 94.0, (3.8, 750): 86.0, (3.8, 1000): 72.0,
    (4.0, 500): 44.0, (4.0, 750): 26.0, (4.0, 1000):  4.0,
    (4.2, 500):  8.0, (4.2, 750):  0.0, (4.2, 1000):  0.0,
}
# H11 results (old formula: (conf+0.1) — too dominant)
h11_old = {
    (3.8, 500): 90.0, (3.8, 750): 80.0, (3.8, 1000): 86.0,
    (4.0, 500): 44.0, (4.0, 750): 26.0, (4.0, 1000): 24.0,
    (4.2, 500):  8.0, (4.2, 750):  2.0, (4.2, 1000):  2.0,
}

results = {m: {} for m in MODES}
ensemble_results = {}

print('=' * 100)
print('H11b — BPR V2 Triple: Confidence vs MFLS vs QuadSurf')
print('  Scoring: exp(-(break-make)/T) × (1 + 0.3·weight)  |  max_flips=200K')
print('=' * 100)
print(f"  {'α':>5} | {'n':>5} | {'S1':>4} | {'Conf%':>6} | {'MFLS%':>6} | "
      f"{'QS%':>6} | {'Ens%':>5} | {'H10b':>5} | {'Best':>8} | Time")
print('  ' + '-' * 90)

for alpha in ALPHAS:
    for n_var in NS:
        m_cls = int(alpha * n_var)
        t0 = time.time()
        s1_count = 0

        # per-mode counters
        mode_ok   = {m: 0 for m in MODES}
        mode_tried = {m: 0 for m in MODES}
        mode_flips = {m: 0 for m in MODES}
        tot_viols = 0
        ens_ok = 0  # ensemble: ANY mode solves
        fc = 0

        for inst in range(N_INST):
            clauses = generate_3sat_instance(n_var, m_cls)
            eng = BSDTGravityV2(n_var, clauses)

            # ── Run gravity ONCE ──
            sf = eng.gravity_flow(steps=STEPS, particles=PARTICLES)
            best_idx, viols = eng.find_best_particle(sf)

            if viols == 0:
                s1_count += 1
                # Perfect from gravity — all modes "solve" it
                for md in MODES:
                    mode_ok[md] += 1
                ens_ok += 1
            else:
                tot_viols += viols
                fc += 1

                # Get initial assignment (same for all modes)
                x_np_base = (sf[best_idx] > 0).cpu().numpy().astype(np.int32)

                # Compute all 3 weight vectors from SAME continuous positions
                w_conf = eng.get_confidence(sf, best_idx)
                w_mfls = eng.get_mfls(sf, best_idx)
                w_qs   = eng.get_quadsurf(sf, best_idx)

                weights = {'confidence': w_conf, 'mfls': w_mfls, 'quadsurf': w_qs}
                any_solved = False

                for md in MODES:
                    mode_tried[md] += 1
                    sol, flips = bpr_repair(
                        eng.clauses_v, eng.clauses_s,
                        x_np_base.copy(),  # fresh copy each mode
                        weights[md],
                        max_flips=MAX_FLIPS, T_init=0.5, T_min=0.01,
                        p_random=0.1, beta=BETA
                    )
                    mode_flips[md] += flips
                    sol_t = torch.tensor(sol, dtype=torch.long, device=device)
                    sat, _ = eng.check_single(sol_t)
                    if sat:
                        mode_ok[md] += 1
                        any_solved = True

                if any_solved:
                    ens_ok += 1

            if (inst + 1) % 10 == 0:
                el = time.time() - t0
                c_pct = mode_ok['confidence'] / (inst+1) * 100
                m_pct = mode_ok['mfls'] / (inst+1) * 100
                q_pct = mode_ok['quadsurf'] / (inst+1) * 100
                print(f'    α={alpha}, n={n_var}: {inst+1}/{N_INST}'
                      f'  C={c_pct:.0f}% M={m_pct:.0f}% Q={q_pct:.0f}%'
                      f'  ({el:.0f}s, ~{el/(inst+1)*N_INST:.0f}s total)')

        elapsed = time.time() - t0
        avg_v = tot_viols / max(fc, 1)
        ws_ref = h10b[(alpha, n_var)]

        # Store results
        best_mode = None
        best_pct = -1
        for md in MODES:
            pct = mode_ok[md] / N_INST * 100
            af = mode_flips[md] // max(mode_tried[md], 1)
            results[md][(alpha, n_var)] = {
                'pct': pct, 'ok': mode_ok[md], 'tried': mode_tried[md],
                'flips': af
            }
            if pct > best_pct:
                best_pct = pct
                best_mode = md

        ens_pct = ens_ok / N_INST * 100
        ensemble_results[(alpha, n_var)] = ens_pct

        c_ = results['confidence'][(alpha, n_var)]['pct']
        m_ = results['mfls'][(alpha, n_var)]['pct']
        q_ = results['quadsurf'][(alpha, n_var)]['pct']
        delta_best = best_pct - ws_ref

        tag = '★' if best_pct >= 95 else ('▲' if delta_best > 0 else
              ('≈' if abs(delta_best) <= 2 else '▼'))

        print(f'  {alpha:5.1f} | {n_var:5d} | {s1_count:3d}  | '
              f'{c_:5.1f}% | {m_:5.1f}% | {q_:5.1f}% | {ens_pct:4.1f}% | '
              f'{ws_ref:4.0f}% | {best_mode[:4]:>4} {delta_best:+.0f}% | '
              f'{elapsed:4.0f}s {tag}')
    print('  ' + '-' * 90)


# ══════════════════════════════════════════════════════════════════════
#  Summary Table
# ══════════════════════════════════════════════════════════════════════
print('\n' + '=' * 100)
print('SUMMARY — Best Mode per (α, n)')
print('=' * 100)
print(f"  {'α':>5} | {'n':>5} | {'H9b':>5} | {'H10b':>5} | {'H11':>5} | "
      f"{'Conf':>5} | {'MFLS':>5} | {'QS':>5} | {'Ens':>5} | {'Best':>8}")
print('  ' + '-' * 80)

for alpha in ALPHAS:
    for n_var in NS:
        c_ = results['confidence'][(alpha, n_var)]['pct']
        m_ = results['mfls'][(alpha, n_var)]['pct']
        q_ = results['quadsurf'][(alpha, n_var)]['pct']
        e_ = ensemble_results[(alpha, n_var)]
        best_val = max(c_, m_, q_)
        best_nm = ['Conf', 'MFLS', 'QS'][[c_, m_, q_].index(best_val)]
        h9  = h9b[(alpha, n_var)]
        h10 = h10b[(alpha, n_var)]
        h11 = h11_old[(alpha, n_var)]

        print(f'  {alpha:5.1f} | {n_var:5d} | {h9:4.0f}% | {h10:4.0f}% | '
              f'{h11:4.0f}% | {c_:4.1f}% | {m_:4.1f}% | {q_:4.1f}% | '
              f'{e_:4.1f}% | {best_nm:>4} {best_val:.0f}%')
    print('  ' + '-' * 80)


# ══════════════════════════════════════════════════════════════════════
#  Charts — 5-way comparison
# ══════════════════════════════════════════════════════════════════════
print('\nGenerating charts...\n')

fig, axes = plt.subplots(1, 3, figsize=(22, 7))
w = 0.15
colors = {
    'H9b': '#e74c3c',
    'H10b (WS)': '#3498db',
    'Confidence': '#2ecc71',
    'MFLS': '#f39c12',
    'QuadSurf': '#9b59b6',
}

for i, n_var in enumerate(NS):
    ax = axes[i]
    x = np.arange(len(ALPHAS))

    bars = [
        ('H9b',        [h9b[(a, n_var)]  for a in ALPHAS]),
        ('H10b (WS)',  [h10b[(a, n_var)] for a in ALPHAS]),
        ('Confidence', [results['confidence'][(a, n_var)]['pct'] for a in ALPHAS]),
        ('MFLS',       [results['mfls'][(a, n_var)]['pct'] for a in ALPHAS]),
        ('QuadSurf',   [results['quadsurf'][(a, n_var)]['pct'] for a in ALPHAS]),
    ]

    for k, (label, vals) in enumerate(bars):
        offset = (k - 2) * w
        ax.bar(x + offset, vals, w, label=label, color=colors[label], alpha=0.85)

    ax.set_xticks(list(x))
    ax.set_xticklabels([str(a) for a in ALPHAS])
    ax.set_xlabel('α (clause ratio)')
    ax.set_ylabel('Solve Rate %')
    ax.set_title(f'n = {n_var}')
    ax.set_ylim(0, 105)
    ax.legend(fontsize=7, loc='upper right')
    ax.grid(axis='y', alpha=0.3)

fig.suptitle('H11b: BPR V2 Triple — Confidence vs MFLS vs QuadSurf\n'
             'score = exp(−ΔE/T) × (1 + 0.3·w)  |  200K flips',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('h11b_triple_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: h11b_triple_results.png')

# ── Flip count comparison ────────────────────────────────────────────
print('\nAverage flip counts (lower = more efficient):')
print(f"  {'α':>5} | {'n':>5} | {'Conf':>8} | {'MFLS':>8} | {'QS':>8}")
print('  ' + '-' * 45)
for alpha in ALPHAS:
    for n_var in NS:
        cf = results['confidence'][(alpha, n_var)]['flips']
        mf = results['mfls'][(alpha, n_var)]['flips']
        qf = results['quadsurf'][(alpha, n_var)]['flips']
        print(f'  {alpha:5.1f} | {n_var:5d} | {cf:8d} | {mf:8d} | {qf:8d}')
    print('  ' + '-' * 45)